In [1]:

from pathlib import Path
import pandas as pd
from scipy.sparse import issparse
from sklearn.utils.multiclass import type_of_target
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from automl import lireDataset, listerDatasets
from optuna import create_study



In [2]:

from sklearn.linear_model import RidgeClassifier, SGDClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.neural_network import MLPClassifier

CLASSIFICATION_MODELS = {
    "RidgeClassifier": {
        "model": RidgeClassifier,
        "sparse": True,
    },
    "SGDClassifier": {
        "model": SGDClassifier,
        "sparse": True,
    },
    "RandomForestClassifier": {
        "model": RandomForestClassifier,
        "sparse": True,
    },
    "ExtraTreesClassifier": {
        "model": ExtraTreesClassifier,
        "sparse": True,
    },
    "HistGradientBoostingClassifier": {
        "model": HistGradientBoostingClassifier,
        "sparse": False,
    },
    "SVC": {
        "model": SVC,
        "sparse": True,
    },
    "LinearSVC": {
        "model": LinearSVC,
        "sparse": True,
    },
    "MultinomialNB": {
        "model": MultinomialNB,
        "sparse": True,
    },
    "GaussianNB": {
        "model": GaussianNB,
        "sparse": False,
    },
}


In [3]:

from sklearn.linear_model import (
    Ridge,
    Lasso,
    ElasticNet,
    SGDRegressor,
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.svm import SVR, LinearSVR

REGRESSION_MODELS = {
    "Ridge":{
        "model":Ridge,
        "sparse":True,
    },
    "Lasso":{
        "model" : Lasso,
        "sparse" : True,
    },
    "ElasticNet": {
        "model": ElasticNet,
        "sparse": True,
    },
    "SGDRegressor": {
        "model": SGDRegressor,
        "sparse": True,
    },
    "DecisionTreeRegressor": {
        "model": DecisionTreeRegressor,
        "sparse": True,
    },
    "RandomForestRegressor": {
        "model": RandomForestRegressor,
        "sparse": True,
    },
    "ExtraTreesRegressor": {
        "model": ExtraTreesRegressor,
        "sparse": True,
    },
    "HistGradientBoostingRegressor": {
        "model": HistGradientBoostingRegressor,
        "sparse": False,
    },
    "SVR": {
        "model": SVR,
        "sparse": True,
    },
    "LinearSVR": {
        "model": LinearSVR,
        "sparse": True,
    },
}


REG_M = {"Ridge":{
        "model":Ridge,
        "sparse":True,
    },
    "Lasso":{
        "model" : Lasso,
        "sparse" : True,
    },"ElasticNet": {
        "model": ElasticNet,
        "sparse": True,
    },
    "RandomForestRegressor": {
        "model": RandomForestRegressor,
        "sparse": True,
    },"DecisionTreeRegressor": {
        "model": RandomForestRegressor,
        "sparse": True,
    },"SGDRegressor": {
        "model": SGDRegressor,
        "sparse": True,
    },"ExtraTreesRegressor": {
        "model": ExtraTreesRegressor,
        "sparse": True,
    }}



In [4]:
def detect_task(y, max_unique_ratio=0.05, max_unique_classification=70):
    y = np.asarray(y).ravel()

    # Retirer les valeurs manquantes
    y = y[pd.notna(y)]

    if len(y) == 0:
        return "unknown"

    # Type de cible reconnu par scikit-learn
    target_type = type_of_target(y)

    if target_type in ("binary", "multiclass"):
        n_samples = len(y)
        n_unique = len(np.unique(y))
        unique_ratio = n_unique / n_samples
        
        if (
            n_unique <= max_unique_classification
            or unique_ratio <= max_unique_ratio
        ):
            return "classification"

        return "regression"

    if target_type in ("continuous", "continuous-multioutput"):
        return "regression"

    if target_type in ("multiclass-multioutput", "multilabel-indicator"):
        return "classification"

    return "unknown"


In [5]:
def load_modele(type):
    if type == "classification":
        return CLASSIFICATION_MODELS
    elif type == "regression":
        return REGRESSION_MODELS
    

In [5]:

from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from scipy import sparse
from sklearn.pipeline import make_pipeline

def objectiveRegression(trial,donne):
  
    model_name = trial.suggest_categorical(
        "model",
        list(REG_M.keys())
    )

    model_class = REG_M[model_name]["model"]
    model = model_class()

    if model_name in ["Ridge","Lasso"]:
        alpha = trial.suggest_float(
            "alpha", 1e-4, 1e4, log=True
        )
        model.set_params(alpha=alpha)

    elif model_name == "ElasticNet":
        alpha = trial.suggest_float("alpha", 1e-4, 1e4, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

        model.set_params(alpha=alpha, l1_ratio=l1_ratio)


    elif model_name == "RandomForestRegressor":
        max_depth = trial.suggest_int("max_depth", 20, 32)
        n_estimators = trial.suggest_int("n_estimators", 400, 500)

        model.set_params(
            max_depth=max_depth,
            n_estimators=n_estimators,
            n_jobs = 32
        )
    elif model_name == "DecisionTreeRegressor":
        max_depth = trial.suggest_int("max_depth", 2, 32)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

        model.set_params(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf
        )


    elif model_name == "SGDRegressor":
        alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
        max_iter = trial.suggest_int("max_iter", 500, 2000)
        tol = trial.suggest_float("tol", 1e-5, 1e-2, log=True)

        model.set_params(
            alpha=alpha, 
            l1_ratio=l1_ratio,
            max_iter=max_iter,
            tol=tol
        )

    elif model_name == "ExtraTreesRegressor":
        max_depth = trial.suggest_int("max_depth", 20, 32)
        n_estimators = trial.suggest_int("n_estimators", 400, 500)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

        model.set_params(
            max_depth=max_depth,
            n_estimators=n_estimators,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf
        )


    if model_name in ["Ridge", "Lasso", "ElasticNet", "SGDRegressor"]:

        scaler = StandardScaler(
            with_mean=not sparse.issparse(donne[0])
        )

        model = make_pipeline(scaler, model)



    scores = cross_val_score(
        model,
        donne[0],
        donne[1],
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()



In [10]:
dossier = Path("/info/corpus/ChallengeMachineLearning/data_B")
df = []
for fichier in sorted(dossier.iterdir()):
    print(fichier.name)
    df.append(pd.read_csv(fichier,sep=r"\s+", header=None))

dd = df[0]
ds = df[1].iloc[:, 0]
dt  =df[2]

print(dd)
print(detect_task(ds))

study = create_study(direction="maximize")
study.optimize(objectiveRegression, n_trials=20)
print("Meilleur score :", study.best_value)
print("Meilleurs paramètres :", study.best_params)

[I 2026-10-16 11:30:57,232] A new study created in memory with name: no-name-626fb520-cf8b-462a-982f-3a8d3de9134d
[I 2026-10-16 11:30:57,270] Trial 0 finished with value: 0.649125327170121 and parameters: {'model': 'Ridge', 'alpha': 0.3363712810748742}. Best is trial 0 with value: 0.649125327170121.
[I 2026-10-16 11:30:57,308] Trial 1 finished with value: 0.5478437522784595 and parameters: {'model': 'Ridge', 'alpha': 5672.816304114903}. Best is trial 0 with value: 0.649125327170121.


data_B.data
data_B.solution
data_B.type
          0   1    2       3    4     5      6     7   8       9       10  \
0     3.5962  29  274 -118.34  550  1597   1220   301  36 -120.89 -117.30   
1     3.9696   9  795 -121.45  379   352   2430    41  25 -117.19 -121.20   
2     2.2417  44  230 -120.38  235   833   3287   188  48 -118.88 -121.48   
3     2.6713  52  300 -117.32  345   867   3694   199  52 -119.19 -117.58   
4     2.1658  44  442 -117.70  244  1947    361   383  29 -118.33 -117.99   
...      ...  ..  ...     ...  ...   ...    ...   ...  ..     ...     ...   
4995  3.5417  10  261 -120.88  356  1400   2406   399  38 -117.93 -119.79   
4996  2.7909  20  319 -116.75  595  5416   1616  1045  26 -119.85 -117.21   
4997  5.1773  23  406 -121.90  410  3529   2547   574  52 -118.39 -122.46   
4998  5.5277  35  231 -118.92  523  2492    894   406  30 -122.04 -121.29   
4999  3.6641  41  427 -122.65  403   976  10535   202  17 -120.97 -119.31   

        11     12   13     14      

[I 2026-10-16 11:31:25,412] Trial 2 finished with value: 0.749598579912359 and parameters: {'model': 'RandomForestRegressor', 'max_depth': 29, 'n_estimators': 421}. Best is trial 2 with value: 0.749598579912359.
[I 2026-10-16 11:31:25,451] Trial 3 finished with value: 0.6491248363151809 and parameters: {'model': 'Ridge', 'alpha': 0.0003156766028950113}. Best is trial 2 with value: 0.749598579912359.
[I 2026-10-16 11:31:54,362] Trial 4 finished with value: 0.7495677918601521 and parameters: {'model': 'RandomForestRegressor', 'max_depth': 28, 'n_estimators': 434}. Best is trial 2 with value: 0.749598579912359.
[I 2026-10-16 11:32:21,735] Trial 5 finished with value: 0.7486693877053525 and parameters: {'model': 'RandomForestRegressor', 'max_depth': 20, 'n_estimators': 415}. Best is trial 2 with value: 0.749598579912359.
[I 2026-10-16 11:32:21,770] Trial 6 finished with value: 0.578944716137898 and parameters: {'model': 'Ridge', 'alpha': 3727.640261981818}. Best is trial 2 with value: 0.74

Meilleur score : 0.751034174014279
Meilleurs paramètres : {'model': 'RandomForestRegressor', 'max_depth': 26, 'n_estimators': 449}


In [6]:
dataSets = listerDatasets(prefix="data")
'''
studies = []
for d in dataSets : 
    dataset = lireDataset(dataSets[1])
    train,test = dataset.split()
    study = create_study(direction="maximize")
    study.optimize(
        lambda trial: objectiveRegression(trial, train),
        n_trials=30
    )
    studies.append(study)
'''
dataset = lireDataset(dataSets[1])
train,test = dataset.split()
study = create_study(direction="maximize")
study.optimize(
    lambda trial: objectiveRegression(trial, train),
    n_trials=30
)

print("Meilleur score :", study.best_value)
print("Meilleurs paramètres :", study.best_params)

[I 2026-11-10 13:45:00,233] A new study created in memory with name: no-name-108bf144-cb82-470e-9fdd-d29516e57283
[I 2026-11-10 13:45:02,236] Trial 0 finished with value: 0.6514191408294915 and parameters: {'model': 'Ridge', 'alpha': 0.006121506951937778}. Best is trial 0 with value: 0.6514191408294915.
[I 2026-11-10 13:45:03,736] Trial 1 finished with value: 0.6514189530208598 and parameters: {'model': 'Lasso', 'alpha': 0.0012305319790791964}. Best is trial 0 with value: 0.6514191408294915.
[I 2026-11-10 13:45:05,215] Trial 2 finished with value: -0.0018541674188113477 and parameters: {'model': 'ElasticNet', 'alpha': 6950.843803616497, 'l1_ratio': 0.7775181196267034}. Best is trial 0 with value: 0.6514191408294915.
[I 2026-11-10 13:45:06,701] Trial 3 finished with value: 0.6260797268768774 and parameters: {'model': 'SGDRegressor', 'alpha': 0.06469478805989484, 'l1_ratio': 0.5686658024491067, 'max_iter': 992, 'tol': 0.001650286304247094}. Best is trial 0 with value: 0.6514191408294915.

Meilleur score : 0.7430938561573799
Meilleurs paramètres : {'model': 'RandomForestRegressor', 'max_depth': 30, 'n_estimators': 447}
